## Semantic Query Caching Simulation — DBpedia (`tab:caching`)

Implements the protocol in `notebooks/caching/README.md` exactly as committed
to in the paper (§Downstream Application):

- **Stream:** held-out SQuARE **test split** anchors, seeded shuffle.
- **Cache:** FIFO over the **K most recent** executed queries (embedding +
  result set). Hit when max cosine ≥ θ → return the cached result, do not
  insert. Miss → "execute" (oracle lookup) and insert.
- **Grid:** K ∈ {50, 100, 200} × θ ∈ {0.7, 0.8, 0.9}.
- **Models:** SparqlCE, TF-IDF, BM25, E5-base-v2,
  Qwen3-Embedding-0.6B, SBERT/MiniLM, SBERT/MPNet.
- **Metrics:** hit rate; result quality = mean answer-set Jaccard between the
  returned (cached) result and the true result, over hits.

Requires the SparqlCE checkpoint `sparqlce/models/final_model_mrr_0.4487.pt`
(not in git — see the main README). Everything
else is in `datasets/`. Outputs `results/caching/caching_dbpedia.json` and
ready-to-paste LaTeX rows for `tab:caching`.


In [1]:
import gc
import os
import sys
import json
import random
from pathlib import Path
from collections import deque

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# ---- Project root (works from notebooks/caching/ or anywhere in the repo) ----
_PROJECT_ROOT = Path.cwd()
while not (_PROJECT_ROOT / "datasets").exists():
    assert _PROJECT_ROOT != _PROJECT_ROOT.parent, "could not locate repo root"
    _PROJECT_ROOT = _PROJECT_ROOT.parent

if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))


# ---- Config ----
TRIPLETS_PATH = (
    _PROJECT_ROOT
    / "datasets"
    / "dbpedia-triplets-dataset.json"
)

SPLITS_PATH = (
    _PROJECT_ROOT
    / "datasets"
    / "splits_dbpedia.json"
)

RESULTS_PATH = (
    _PROJECT_ROOT
    / "datasets"
    / "result_sets_dbpedia.json"
)

SPARQLCE_CKPT = (
    _PROJECT_ROOT
    / "sparqlce"
    / "models"
    / "final_model_mrr_0.4487.pt"
)

OUT_DIR = (
    _PROJECT_ROOT
    / "results"
    / "caching"
)


SEED = 42

K_GRID = [
    50,
    100,
    200,
]

THETA_GRID = [
    0.7,
    0.8,
    0.9,
]

# (K, theta) row reported in the main table
HEADLINE = (100, 0.8)


DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    f"project root: {_PROJECT_ROOT}\n"
    f"device: {DEVICE}"
)

project root: /workspace/SPARQL-Embeddings
device: cuda


/workspace/.conda/envs/sparql/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# def normalize_query(q: str) -> str:
#     """Normalize formatting for exact oracle lookup only."""
#     return " ".join(q.split())


# with open(TRIPLETS_PATH, encoding="utf-8") as f:
#     triplets = json.load(f)

# with open(SPLITS_PATH, encoding="utf-8") as f:
#     splits = json.load(f)

# with open(RESULTS_PATH, encoding="utf-8") as f:
#     raw_result_sets = json.load(f)


# # Keep only non-empty answer sets and normalize query keys.
# result_sets = {
#     normalize_query(q): frozenset(rows)
#     for q, rows in raw_result_sets.items()
#     if rows
# }


# test_anchors = [
#     triplets[i]["anchor_query"]
#     for i in splits["test_idx"]
# ]

# train_anchors = [
#     triplets[i]["anchor_query"]
#     for i in splits["train_idx"]
# ]


# # Keep the ORIGINAL query strings in the stream for embedding.
# # Use normalized forms only to find oracle results.
# stream = [
#     q
#     for q in test_anchors
#     if normalize_query(q) in result_sets
# ]

# dropped = len(test_anchors) - len(stream)

# random.seed(SEED)
# random.shuffle(stream)

# print(
#     f"test anchors: {len(test_anchors)} | "
#     f"in stream: {len(stream)} | "
#     f"dropped (missing/empty oracle result): {dropped}"
# )


# def jaccard(a: frozenset, b: frozenset) -> float:
#     union = len(a | b)
#     return len(a & b) / union if union else 0.0


In [3]:
def normalize_query(q: str) -> str:
    """
    Normalize query formatting for exact oracle lookup only.

    The original query string is still used for embedding.
    """
    return " ".join(q.split())


def freeze_json(value):
    """
    Convert a JSON-compatible value into a recursively hashable form.

    This allows result rows represented as lists or dictionaries to be
    stored safely in a frozenset for answer-set Jaccard computation.
    """
    if isinstance(value, dict):
        return tuple(
            sorted(
                (
                    key,
                    freeze_json(item),
                )
                for key, item in value.items()
            )
        )

    if isinstance(value, list):
        return tuple(
            freeze_json(item)
            for item in value
        )

    return value


with open(
    TRIPLETS_PATH,
    encoding="utf-8",
) as f:
    triplets = json.load(f)


with open(
    SPLITS_PATH,
    encoding="utf-8",
) as f:
    splits = json.load(f)


with open(
    RESULTS_PATH,
    encoding="utf-8",
) as f:
    raw_result_sets = json.load(f)


# ---------------------------------------------------------------------------
# Build oracle lookup.
#
# Query keys are normalized only for exact oracle lookup.
# Result rows are converted into hashable canonical Python structures.
#
# A normalization collision with different result sets is treated as an error
# rather than silently overwriting one oracle entry with another.
# ---------------------------------------------------------------------------
result_sets = {}

for raw_query, rows in raw_result_sets.items():

    if not rows:
        continue

    query_key = normalize_query(raw_query)

    answer_set = frozenset(
        freeze_json(row)
        for row in rows
    )

    if (
        query_key in result_sets
        and result_sets[query_key] != answer_set
    ):
        raise ValueError(
            "Conflicting oracle result sets after query normalization:\n"
            f"{query_key}"
        )

    result_sets[query_key] = answer_set


# ---------------------------------------------------------------------------
# Build train/test anchor collections from the committed split.
# ---------------------------------------------------------------------------
test_anchors = [
    triplets[i]["anchor_query"]
    for i in splits["test_idx"]
]

train_anchors = [
    triplets[i]["anchor_query"]
    for i in splits["train_idx"]
]


# ---------------------------------------------------------------------------
# Keep ORIGINAL query strings in the stream for embedding.
#
# Normalized query strings are used only for oracle lookup.
# ---------------------------------------------------------------------------
stream = [
    q
    for q in test_anchors
    if normalize_query(q) in result_sets
]

dropped = (
    len(test_anchors)
    - len(stream)
)


# ---------------------------------------------------------------------------
# Shuffle the occurrence-level stream deterministically.
#
# Important:
# This deliberately does NOT deduplicate repeated anchors. If the same query
# occurs multiple times in the held-out workload, those occurrences remain
# separate stream events.
# ---------------------------------------------------------------------------
random.seed(SEED)
random.shuffle(stream)


# ---------------------------------------------------------------------------
# Diagnostics only. These values do not change the experiment.
# ---------------------------------------------------------------------------
stream_unique_queries = len(
    {
        normalize_query(q)
        for q in stream
    }
)

duplicate_stream_events = (
    len(stream)
    - stream_unique_queries
)


print(
    f"test anchors: {len(test_anchors)} | "
    f"in stream: {len(stream)} | "
    f"dropped (missing/empty oracle result): {dropped}"
)

print(
    f"unique normalized queries in stream: "
    f"{stream_unique_queries} | "
    f"repeated stream events: "
    f"{duplicate_stream_events}"
)


def jaccard(
    a: frozenset,
    b: frozenset,
) -> float:
    """
    Answer-set Jaccard similarity.
    """
    union = len(a | b)

    return (
        len(a & b) / union
        if union
        else 0.0
    )

test anchors: 472 | in stream: 396 | dropped (missing/empty oracle result): 76
unique normalized queries in stream: 396 | repeated stream events: 0


In [6]:
# ============================================================================
# 2. Embed the stream once per model
#
# Every representation is L2-normalised so that the cache lookup can continue
# to use a common cosine-similarity threshold grid.
# ============================================================================

from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer

stream_embeddings = {}


def clear_model(model):
    """Release a transformer model and clear accelerator memory."""
    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ---------------------------------------------------------------------------
# SparqlCE — trained checkpoint
# ---------------------------------------------------------------------------
from sparqlce.embedder import SPARQLEmbedder

sparqlce = SPARQLEmbedder(
    str(SPARQLCE_CKPT)
)

embs = []

for i in tqdm(
    range(0, len(stream), 32),
    desc="SparqlCE",
):
    embs.append(
        sparqlce.encode(
            stream[i : i + 32]
        )
    )

stream_embeddings["SparqlCE"] = normalize(
    np.vstack(embs),
    norm="l2",
    axis=1,
).astype(
    np.float32,
    copy=False,
)

del sparqlce
del embs
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ---------------------------------------------------------------------------
# TF-IDF
#
# Vocabulary and IDF statistics are fit on TRAIN anchors only.
# ---------------------------------------------------------------------------
tfidf = TfidfVectorizer(
    norm="l2",
).fit(
    train_anchors
)

stream_embeddings["TF-IDF"] = (
    tfidf
    .transform(stream)
    .toarray()
    .astype(np.float32, copy=False)
)


# ---------------------------------------------------------------------------
# BM25
#
# The cache simulator requires a bounded cosine similarity because theta is
# shared across all baselines. We therefore build Okapi-BM25 term-weight
# vectors using TRAIN-anchor corpus statistics, then L2-normalise each vector.
# Cache lookup remains a cosine comparison between query representations.
# ---------------------------------------------------------------------------
class BM25Vectorizer:
    """Okapi-BM25 term weighting with optional row normalisation."""

    def __init__(
        self,
        *,
        k1=1.5,
        b=0.75,
        norm="l2",
        **count_vectorizer_kwargs,
    ):
        self.k1 = float(k1)
        self.b = float(b)
        self.norm = norm
        self.vectorizer = CountVectorizer(
            **count_vectorizer_kwargs
        )

    def fit(self, documents):
        counts = (
            self.vectorizer
            .fit_transform(documents)
            .tocsr()
            .astype(np.float32)
        )

        n_documents = counts.shape[0]

        if n_documents == 0:
            raise ValueError(
                "BM25Vectorizer cannot be fit on an empty corpus."
            )

        document_frequency = np.asarray(
            (counts > 0).sum(axis=0)
        ).ravel()

        # Robertson/Sparck Jones IDF with the common +1 stabilisation.
        self.idf_ = np.log(
            1.0
            + (
                n_documents
                - document_frequency
                + 0.5
            )
            / (
                document_frequency
                + 0.5
            )
        ).astype(np.float32)

        document_lengths = np.asarray(
            counts.sum(axis=1)
        ).ravel()

        self.avg_document_length_ = float(
            document_lengths.mean()
        )

        if self.avg_document_length_ <= 0:
            raise ValueError(
                "BM25Vectorizer received only empty documents."
            )

        return self

    def transform(self, documents):
        counts = (
            self.vectorizer
            .transform(documents)
            .tocoo()
            .astype(np.float32)
        )

        document_lengths = np.asarray(
            counts.tocsr().sum(axis=1)
        ).ravel()

        length_factor = (
            1.0
            - self.b
            + self.b
            * (
                document_lengths
                / self.avg_document_length_
            )
        )

        denominator = (
            counts.data
            + self.k1
            * length_factor[counts.row]
        )

        weighted_data = (
            self.idf_[counts.col]
            * counts.data
            * (self.k1 + 1.0)
            / denominator
        )

        weighted = sparse.csr_matrix(
            (
                weighted_data,
                (
                    counts.row,
                    counts.col,
                ),
            ),
            shape=counts.shape,
            dtype=np.float32,
        )

        if self.norm is not None:
            weighted = normalize(
                weighted,
                norm=self.norm,
                axis=1,
                copy=False,
            )

        return weighted

    def fit_transform(self, documents):
        return (
            self
            .fit(documents)
            .transform(documents)
        )


bm25 = BM25Vectorizer(
    k1=1.5,
    b=0.75,
    norm="l2",
).fit(
    train_anchors
)

stream_embeddings["BM25"] = (
    bm25
    .transform(stream)
    .toarray()
    .astype(np.float32, copy=False)
)


# ---------------------------------------------------------------------------
# Dense text-embedding baselines
# ---------------------------------------------------------------------------
model_specs = [
    {
        "name": "E5-base-v2",
        "model_id": "intfloat/e5-base-v2",
        "texts": [
            f"query: {q}"
            for q in stream
        ],
        "batch_size": 32,
    },
    {
        "name": "Qwen3-Embedding-0.6B",
        "model_id": "Qwen/Qwen3-Embedding-0.6B",
        "texts": stream,
        "batch_size": 8,
    },
    {
        "name": "SBERT/MiniLM",
        "model_id": "sentence-transformers/all-MiniLM-L6-v2",
        "texts": stream,
        "batch_size": 64,
    },
    {
        "name": "SBERT/MPNet",
        "model_id": "sentence-transformers/all-mpnet-base-v2",
        "texts": stream,
        "batch_size": 32,
    },
]


for spec in model_specs:
    model = SentenceTransformer(
        spec["model_id"],
        device=DEVICE,
    )

    encoded = model.encode(
        spec["texts"],
        convert_to_numpy=True,
        normalize_embeddings=False,
        show_progress_bar=True,
        batch_size=spec["batch_size"],
    ).astype(
        np.float32,
        copy=False,
    )

    # Explicit post-normalisation avoids small model/dtype-dependent norm drift
    # (observed for Qwen3-Embedding-0.6B despite normalize_embeddings=True).
    stream_embeddings[
        spec["name"]
    ] = normalize(
        encoded,
        norm="l2",
        axis=1,
        copy=False,
    ).astype(
        np.float32,
        copy=False,
    )

    del encoded
    clear_model(model)


# ============================================================================
# Fine-tuned Qwen3-Embedding-0.6B — SQuARE DBpedia LoRA adapter
#
# Uses exactly the same:
#   - base model
#   - tokenizer padding direction
#   - max sequence length
#   - last-token pooling
#   - L2 normalization
# as the fine-tuning notebook.
# ============================================================================

from transformers import AutoModel, AutoTokenizer
from peft import PeftModel
import torch.nn.functional as F


FT_QWEN_BASE = "Qwen/Qwen3-Embedding-0.6B"

FT_QWEN_ADAPTER = (
    _PROJECT_ROOT
    / "scripts"
    / "qwen_ft"
    / "qwen3_square_dbpedia"
    / "best_adapter"
)

FT_QWEN_NAME = "Qwen3-Embedding-0.6B (fine-tuned)"
FT_QWEN_BATCH_SIZE = 8
FT_QWEN_MAX_LENGTH = 512


if not FT_QWEN_ADAPTER.exists():
    raise FileNotFoundError(
        f"Fine-tuned Qwen adapter not found: {FT_QWEN_ADAPTER}"
    )

if not (FT_QWEN_ADAPTER / "adapter_config.json").exists():
    raise FileNotFoundError(
        f"Missing adapter_config.json in {FT_QWEN_ADAPTER}"
    )

if not (FT_QWEN_ADAPTER / "adapter_model.safetensors").exists():
    raise FileNotFoundError(
        f"Missing adapter_model.safetensors in {FT_QWEN_ADAPTER}"
    )


def qwen_last_token_pool(
    last_hidden_states,
    attention_mask,
):
    """
    Last-token pooling used during fine-tuning.

    Handles both left-padded and right-padded batches.
    """
    left_padding = (
        attention_mask[:, -1].sum()
        == attention_mask.shape[0]
    )

    if left_padding:
        return last_hidden_states[:, -1]

    sequence_lengths = (
        attention_mask.sum(dim=1)
        - 1
    )

    batch_indices = torch.arange(
        last_hidden_states.shape[0],
        device=last_hidden_states.device,
    )

    return last_hidden_states[
        batch_indices,
        sequence_lengths,
    ]


# Load the tokenizer saved alongside the adapter.
ft_qwen_tokenizer = AutoTokenizer.from_pretrained(
    str(FT_QWEN_ADAPTER),
    padding_side="left",
)

if ft_qwen_tokenizer.pad_token_id is None:
    ft_qwen_tokenizer.pad_token = (
        ft_qwen_tokenizer.eos_token
    )


# Match the dtype used during fine-tuning.
ft_qwen_dtype = (
    torch.bfloat16
    if torch.cuda.is_available()
    else torch.float32
)


# Load frozen base model, then attach the trained LoRA adapter.
ft_qwen_base = AutoModel.from_pretrained(
    FT_QWEN_BASE,
    torch_dtype=ft_qwen_dtype,
)

ft_qwen_model = PeftModel.from_pretrained(
    ft_qwen_base,
    str(FT_QWEN_ADAPTER),
    is_trainable=False,
)

ft_qwen_model = ft_qwen_model.to(DEVICE)
ft_qwen_model.eval()


@torch.inference_mode()
def encode_finetuned_qwen(
    queries,
    *,
    batch_size=FT_QWEN_BATCH_SIZE,
    max_length=FT_QWEN_MAX_LENGTH,
):
    embedding_batches = []

    for start in tqdm(
        range(0, len(queries), batch_size),
        desc=FT_QWEN_NAME,
    ):
        batch_queries = queries[
            start : start + batch_size
        ]

        tokens = ft_qwen_tokenizer(
            batch_queries,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )

        tokens = {
            key: value.to(DEVICE)
            for key, value in tokens.items()
        }

        output = ft_qwen_model(
            input_ids=tokens["input_ids"],
            attention_mask=tokens["attention_mask"],
        )

        embeddings = qwen_last_token_pool(
            output.last_hidden_state,
            tokens["attention_mask"],
        )

        # Convert to float32 before normalization, matching training.
        embeddings = F.normalize(
            embeddings.float(),
            p=2,
            dim=1,
        )

        embedding_batches.append(
            embeddings.cpu().numpy()
        )

    return np.vstack(
        embedding_batches
    ).astype(
        np.float32,
        copy=False,
    )


stream_embeddings[
    FT_QWEN_NAME
] = encode_finetuned_qwen(
    stream
)


# Verify that the embeddings are finite and unit-normalized.
ft_embeddings = stream_embeddings[
    FT_QWEN_NAME
]

if not np.isfinite(ft_embeddings).all():
    raise ValueError(
        f"{FT_QWEN_NAME} produced NaN or infinite values."
    )

ft_norms = np.linalg.norm(
    ft_embeddings,
    axis=1,
)

if not np.allclose(
    ft_norms,
    1.0,
    atol=1e-5,
):
    raise ValueError(
        f"{FT_QWEN_NAME} embeddings are not L2-normalized. "
        f"Norm range: {ft_norms.min():.6f}–{ft_norms.max():.6f}"
    )


print(
    f"{FT_QWEN_NAME}: "
    f"{stream_embeddings[FT_QWEN_NAME].shape}"
)


# Release GPU memory after encoding.
del ft_qwen_model
del ft_qwen_base
del ft_qwen_tokenizer
del ft_embeddings
del ft_norms

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ---------------------------------------------------------------------------
# Sanity checks
# ---------------------------------------------------------------------------
for name, E in stream_embeddings.items():
    if E.shape[0] != len(stream):
        raise ValueError(
            f"{name}: expected {len(stream)} rows, got {E.shape[0]}"
        )

    row_norms = np.linalg.norm(
        E,
        axis=1,
    )

    nonzero_rows = row_norms > 0

    if np.any(nonzero_rows) and not np.allclose(
        row_norms[nonzero_rows],
        1.0,
        atol=1e-5,
    ):
        raise ValueError(
            f"{name}: non-zero rows are not L2-normalised."
        )

    print(
        f"{name:>24}: {E.shape}"
    )


Loading model from /workspace/SPARQL-Embeddings/sparqlce/models/final_model_mrr_0.4487.pt...


Batches: 100%|██████████| 13/13 [00:00<00:00, 14.76it/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Qwen3-Embedding-0.6B (fine-tuned): 100%|██████████| 50/50 [00:02<00:00, 16.68it/s]


Qwen3-Embedding-0.6B (fine-tuned): (396, 1024)
                SparqlCE: (396, 128)
                  TF-IDF: (396, 3670)
                    BM25: (396, 3670)
              E5-base-v2: (396, 768)
    Qwen3-Embedding-0.6B: (396, 1024)
            SBERT/MiniLM: (396, 384)
             SBERT/MPNet: (396, 768)
Qwen3-Embedding-0.6B (fine-tuned): (396, 1024)


In [7]:
# ============================================================================
# 3. Cache simulation
#
# Protocol:
#   - FIFO over the K most recent EXECUTED queries.
#   - Hit when max cosine similarity >= theta.
#   - On a hit: return the cached result and do not modify the cache.
#   - On a miss: use the oracle result and insert the query/result pair.
# ============================================================================

def simulate(
    stream,
    E,
    result_sets,
    K,
    theta,
):
    """
    Run one pass over the query stream.

    E[i] is the L2-normalised embedding of stream[i].
    """

    if len(stream) == 0:
        raise ValueError(
            "Cannot simulate an empty query stream."
        )

    if len(E) != len(stream):
        raise ValueError(
            "Embedding matrix length does not match stream length: "
            f"{len(E)} != {len(stream)}"
        )

    if K <= 0:
        raise ValueError(
            f"K must be positive, got {K}."
        )


    # Each entry is:
    #   (query_embedding, oracle_result_set)
    #
    # maxlen=K gives FIFO eviction automatically.
    cache = deque(
        maxlen=K
    )

    hits = 0
    qualities = []


    for i, q in enumerate(stream):

        R_true = result_sets[
            normalize_query(q)
        ]


        # -------------------------------------------------------------------
        # Cache lookup
        # -------------------------------------------------------------------
        if cache:

            cached_embeddings = np.stack(
                [
                    entry[0]
                    for entry in cache
                ]
            )

            sims = (
                cached_embeddings
                @ E[i]
            )

            best = int(
                np.argmax(sims)
            )


            # ---------------------------------------------------------------
            # Hit:
            #
            # Serve the nearest cached result.
            # Do not execute.
            # Do not insert.
            # Do not refresh cache position.
            # ---------------------------------------------------------------
            if float(sims[best]) >= theta:

                R_cached = cache[best][1]

                hits += 1

                qualities.append(
                    jaccard(
                        R_cached,
                        R_true,
                    )
                )

                continue


        # -------------------------------------------------------------------
        # Miss:
        #
        # "Execute" through the offline oracle and insert into FIFO cache.
        # -------------------------------------------------------------------
        cache.append(
            (
                E[i],
                R_true,
            )
        )


    n_queries = len(stream)

    return {
        "hit_rate": hits / n_queries,
        "result_quality": (
            float(np.mean(qualities))
            if qualities
            else None
        ),
        "n_hits": hits,
        "n_misses": n_queries - hits,
        "n_queries": n_queries,
    }


all_results = {}

for name, E in stream_embeddings.items():

    all_results[name] = {}

    for K in K_GRID:
        for theta in THETA_GRID:

            r = simulate(
                stream=stream,
                E=E,
                result_sets=result_sets,
                K=K,
                theta=theta,
            )

            all_results[name][
                f"K={K},theta={theta}"
            ] = r

    print(
        f"done: {name}"
    )

done: SparqlCE
done: TF-IDF
done: BM25
done: E5-base-v2
done: Qwen3-Embedding-0.6B
done: SBERT/MiniLM
done: SBERT/MPNet
done: Qwen3-Embedding-0.6B (fine-tuned)


In [8]:
# ============================================================================
# 4. Save the full grid + display
# ============================================================================
OUT_DIR.mkdir(parents=True, exist_ok=True)
payload = {
    "kg": "dbpedia",
    "seed": SEED,
    "stream_length": len(stream),
    "dropped": dropped,
    "grid": {"K": K_GRID, "theta": THETA_GRID},
    "results": all_results,
}
out_json = OUT_DIR / "caching_dbpedia.json"
with open(out_json, "w") as f:
    json.dump(payload, f, indent=2)
print(f"saved {out_json}\n")

rows = []
for name, grid in all_results.items():
    for cfg, r in grid.items():
        K, theta = cfg.replace("K=", "").split(",theta=")
        rows.append({"model": name, "K": int(K), "theta": float(theta),
                     "hit_rate": r["hit_rate"], "result_quality": r["result_quality"]})
df = pd.DataFrame(rows)

print("Hit rate:")
print(df.pivot_table(index="model", columns=["K", "theta"], values="hit_rate").round(3))
print("\nResult quality (mean Jaccard over hits):")
print(df.pivot_table(index="model", columns=["K", "theta"], values="result_quality").round(3))

saved /workspace/SPARQL-Embeddings/results/caching/caching_dbpedia.json

Hit rate:
K                                    50                   100                \
theta                                0.7    0.8    0.9    0.7    0.8    0.9   
model                                                                         
BM25                               0.664  0.641  0.621  0.694  0.674  0.646   
E5-base-v2                         0.997  0.997  0.970  0.997  0.997  0.970   
Qwen3-Embedding-0.6B               0.674  0.207  0.008  0.730  0.298  0.013   
Qwen3-Embedding-0.6B (fine-tuned)  0.740  0.508  0.101  0.788  0.601  0.159   
SBERT/MPNet                        0.977  0.939  0.795  0.977  0.939  0.826   
SBERT/MiniLM                       0.972  0.955  0.874  0.972  0.955  0.874   
SparqlCE                           0.831  0.604  0.253  0.843  0.687  0.328   
TF-IDF                             0.795  0.753  0.644  0.826  0.788  0.662   

K                                    200       

In [10]:
# ============================================================================
# 5. LaTeX rows for tab:caching
# ============================================================================

K_h, theta_h = HEADLINE


order = [
    "SparqlCE",
    "TF-IDF",
    "BM25",
    "E5-base-v2",
    "Qwen3-Embedding-0.6B",
    "Qwen3-Embedding-0.6B (fine-tuned)",
    "SBERT/MiniLM",
    "SBERT/MPNet",
]


latex_names = {
    "SparqlCE": r"\textsc{SparqlCE}",
    "TF-IDF": "TF-IDF",
    "BM25": "BM25",
    "E5-base-v2": "E5-base-v2",
    "Qwen3-Embedding-0.6B": "Qwen3-Embedding-0.6B",
    "Qwen3-Embedding-0.6B (fine-tuned)": "Qwen3-Embedding-0.6B (fine-tuned)",
    "SBERT/MiniLM": "SBERT/MiniLM",
    "SBERT/MPNet": "SBERT/MPNet",
}


def latex_quality(value):
    """
    Format result quality for a LaTeX table.

    Undefined quality occurs when a configuration produces zero hits.
    """
    return (
        f"{value:.3f}"
        if value is not None
        else "--"
    )


print(
    f"% tab:caching — headline operating point "
    f"(K={K_h}, theta={theta_h}), "
    f"full grid in "
    f"results/caching/caching_dbpedia.json"
)


for name in order:

    r = all_results[name][
        f"K={K_h},theta={theta_h}"
    ]

    quality_str = latex_quality(
        r["result_quality"]
    )

    print(
        f"{latex_names[name]:<24} "
        f"& $({K_h}, {theta_h})$ "
        f"& {r['hit_rate']:.3f} "
        f"& {quality_str} \\\\"
    )


# ---------------------------------------------------------------------------
# Headline sentence draft
# ---------------------------------------------------------------------------
best = all_results[
    "SparqlCE"
][
    f"K={K_h},theta={theta_h}"
]


alts = {
    name: all_results[name][
        f"K={K_h},theta={theta_h}"
    ]
    for name in order[1:]
}


valid_alts = {
    name: result
    for name, result in alts.items()
    if result["result_quality"] is not None
}


print(
    "\n% headline sentence draft:"
)


if best["result_quality"] is None:

    print(
        f"% At K={K_h}, theta={theta_h}, "
        f"SparqlCE produces no cache hits, so "
        f"result quality is undefined."
    )


elif not valid_alts:

    print(
        f"% At K={K_h}, theta={theta_h}, "
        f"SparqlCE serves "
        f"{best['hit_rate']:.0%} of the stream "
        f"from cache with "
        f"{best['result_quality']:.2f} result quality. "
        f"No baseline produces a cache hit at this "
        f"operating point."
    )


else:

    best_base = max(
        valid_alts,
        key=lambda name: (
            valid_alts[name][
                "result_quality"
            ]
        ),
    )

    print(
        f"% At K={K_h}, theta={theta_h}, "
        f"SparqlCE serves "
        f"{best['hit_rate']:.0%} of the stream "
        f"from cache with "
        f"{best['result_quality']:.2f} result quality, "
        f"vs. "
        f"{valid_alts[best_base]['result_quality']:.2f} "
        f"for the best baseline "
        f"({best_base})."
    )


% tab:caching — headline operating point (K=100, theta=0.8), full grid in results/caching/caching_dbpedia.json
\textsc{SparqlCE}        & $(100, 0.8)$ & 0.687 & 0.264 \\
TF-IDF                   & $(100, 0.8)$ & 0.788 & 0.069 \\
BM25                     & $(100, 0.8)$ & 0.674 & 0.086 \\
E5-base-v2               & $(100, 0.8)$ & 0.997 & 0.017 \\
Qwen3-Embedding-0.6B     & $(100, 0.8)$ & 0.298 & 0.133 \\
Qwen3-Embedding-0.6B (fine-tuned) & $(100, 0.8)$ & 0.601 & 0.307 \\
SBERT/MiniLM             & $(100, 0.8)$ & 0.955 & 0.036 \\
SBERT/MPNet              & $(100, 0.8)$ & 0.939 & 0.048 \\

% headline sentence draft:
% At K=100, theta=0.8, SparqlCE serves 69% of the stream from cache with 0.26 result quality, vs. 0.31 for the best baseline (Qwen3-Embedding-0.6B (fine-tuned)).


### Notes

- BM25 uses L2-normalized Okapi-BM25 term-weight vectors so that it can be
  evaluated with the same cosine-threshold cache protocol as the embedding
  baselines.
- The oracle (`result_sets_dbpedia.json`) replaces live endpoint execution, so
  the simulation is deterministic and offline.
- Rerunning with a different stream order: change `SEED` — report the default
  seed-42 numbers in the paper.
- Wikidata twin requires `datasets/result_sets_wikidata.json` (not in the
  repo yet).
